# Experiment: Ablation Study — Data & RAG Setup

**Ziel:** Datenbasis für die Ablation Study zum Monolith-RAG aufbauen.  
**Unternehmen:** AAPL, MSFT, AMZN, META  
**Fiscal Years:** FY2022 + FY2024  
**Filings:** 8 × 10-K

---

## 1. Setup

In [ ]:
import logging
from src.common import download_filing, load_processed_filing, setup_logging

setup_logging(logging.INFO)

## 2. Download: 8 × 10-K Filings

Retry-Logik ist jetzt direkt in `ingestion.py` via `tenacity` eingebaut.  
Download-Kaskade: edgartools → datamule/secsgml → httpx direkt.

| Ticker | FY2022 | FY2024 |
|--------|--------|--------|
| AAPL   | ⬜     | ⬜     |
| MSFT   | ⬜     | ⬜     |
| AMZN   | ⬜     | ⬜     |
| META   | ⬜     | ⬜     |

In [ ]:
# Define ablation study dataset
ABLATION_FILINGS = [
    ("AAPL",  2022),
    ("AAPL",  2024),
    ("MSFT",  2022),
    ("MSFT",  2024),
    ("AMZN",  2022),
    ("AMZN",  2024),
    ("GOOGL",  2022),
    ("GOOGL",  2024),
]

In [ ]:
# Download all filings
# Retry-Logik (tenacity) ist in ingestion.py eingebaut — kein manuelles Retry nötig.

results = {}
failed = []

for i, (ticker, fy) in enumerate(ABLATION_FILINGS):
    key = f"{ticker}_FY{fy}"
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(ABLATION_FILINGS)}] Downloading {key}...")
    print(f"{'='*60}")

    try:
        filing = download_filing(ticker, fiscal_year=fy)
        results[key] = filing
        print(f"  ✓ {key}: {len(filing.sections)} sections, {len(filing.full_text):,} chars")
    except Exception as e:
        failed.append(key)
        print(f"  ✗ {key}: FAILED — {e}")

print(f"\n\n{'='*60}")
print(f"Downloaded {len(results)}/{len(ABLATION_FILINGS)} filings")
if failed:
    print(f"Failed: {', '.join(failed)}")

## 3. Übersicht: Heruntergeladene Filings

In [ ]:
# Summary table
print(f"{'Key':<15} | {'Company':<30} | {'Filed':<12} | {'FY End':<12} | {'Sections':>8} | {'Chars':>10}")
print("-" * 95)
for key, f in results.items():
    print(
        f"{key:<15} | {f.metadata.company_name:<30} | "
        f"{f.metadata.filing_date:<12} | {f.metadata.fiscal_year_end:<12} | "
        f"{len(f.sections):>8} | {len(f.full_text):>10,}"
    )

In [ ]:
# Section breakdown for one filing (spot check)
sample = list(results.values())[0] if results else None
if sample:
    print(f"\nSections in {sample.metadata.ticker} FY{sample.metadata.fiscal_year_end[:4]}:")
    print("-" * 60)
    for section_name, content in sample.sections.items():
        print(f"  {section_name:45s} | {len(content):>8,} chars")

## 4. Validierung: Datenqualität prüfen

In [ ]:
# Validate: each filing should have at least 3 key sections
REQUIRED_SECTIONS = {"Business", "Risk Factors", "MD&A"}

print(f"{'Key':<15} | {'Sections':>8} | {'Missing':>40}")
print("-" * 70)

all_ok = True
for key, f in results.items():
    present = set(f.sections.keys())
    missing = REQUIRED_SECTIONS - present
    status = '✓' if not missing else '⚠'
    missing_str = ', '.join(missing) if missing else '—'
    print(f"  {status} {key:<13} | {len(f.sections):>8} | {missing_str:>40}")
    if missing:
        all_ok = False

print()
if all_ok:
    print("✅ Alle Filings haben die 3 Kern-Sections.")
else:
    print("⚠️  Einige Filings haben fehlende Sections — prüfe die Logs oben.")